# Chapter 6: Data Quality and Governance

This notebook complements Chapter 6 by turning control questions into asset-by-asset operating decisions.

The current chapter treats prompt, context, and generated-output governance as part of the same control plane as freshness, lineage, ownership, security, and retention. Use the retail ranking track for feature and benchmark controls. Use the support-assistant track for prompt versions, retrieved context, tool auditability, and output retention.

In [ ]:
from pathlib import Path

def find_companion_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not find companion repository root.")


def ensure_path_exists(path: Path, kind: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected {kind}: {path}. "
            "Make sure you are running this notebook from inside the companion repository."
        )
    return path

NOTEBOOK_DIR = Path.cwd()
COMPANION_ROOT = find_companion_root(NOTEBOOK_DIR)
print("Companion root located successfully.")

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location

import pandas as pd

module_path = ensure_path_exists(
    COMPANION_ROOT / "code" / "quality" / "control_assessment.py",
    "helper module",
)
spec = spec_from_file_location("control_assessment", module_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load helper module spec from {module_path}")
control_assessment = module_from_spec(spec)
spec.loader.exec_module(control_assessment)

load_quality_assets = control_assessment.load_quality_assets
summarize_control_posture = control_assessment.summarize_control_posture
assess_controls = control_assessment.assess_controls
find_high_priority_assets = control_assessment.find_high_priority_assets

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

### Why the Setup Code Matters

The helper code turns Chapter 6's control language into repeatable assessments. The notebook is not trying to automate governance away. It is trying to make severity, ownership, freshness, benchmark drift, retrieval authorization, and prompt-trace controls legible enough to compare.

## Part 1: Load Asset Control Scenarios

The control table now includes classic feature assets, benchmark assets, retrieval indexes, and embedding stores.


In [ ]:
asset_path = ensure_path_exists(
    COMPANION_ROOT / "datasets" / "quality" / "sample_quality_assets.csv",
    "dataset",
)
assets_df = load_quality_assets(asset_path)
assets_df[[
    "asset_name",
    "asset_type",
    "criticality",
    "freshness_delay_hours",
    "retrieval_freshness_hours",
    "benchmark_owner_defined",
]].sort_values("asset_name").reset_index(drop=True)

### What to Notice in the Asset Table

Look for assets that are simultaneously sensitive, model-facing, and weakly owned. Those combinations usually create the hardest incidents because the platform has both technical and governance uncertainty at the same time.


## Part 2: Summarize Control Posture

This summary now shows how many assets are model-facing, how many benchmark assets lack ownership, and how many retrieval assets are stale enough to affect downstream behavior.


In [ ]:
summary_df = summarize_control_posture(assets_df)
summary_df


### Interpreting the Control Summary

The summary should help you see where the control plane is thinning out. A high count of stale assets, low-lineage assets, or unowned benchmark assets means problems will surface first as slow investigations rather than obvious system crashes.


## Part 3: Assess Control Risk

The helper attaches a total risk score, a primary gap, and a more specific observability gap for each asset.


In [ ]:
assessment_df = assess_controls(assets_df)
assessment_df[[
    "asset_name",
    "asset_type",
    "priority",
    "primary_gap",
    "observability_gap",
    "recommended_action",
]].sort_values(["priority", "asset_name"]).reset_index(drop=True)


### Interpreting the Risk Assessment

Each assessment row is a decision prompt. The question is not whether the score is high in the abstract. The question is which gap deserves action first: freshness, lineage, ownership, access review, benchmark discipline, or retrieval-state observability.


## Part 4: Focus on Model-Facing Observability

Model-facing observability is its own control idea. This view isolates the assets where freshness, benchmark drift, chunking change, prompt edits, or retrieval-state staleness can quietly change downstream behavior before a hard outage appears.

In [ ]:
model_facing_df = assessment_df.loc[
    assessment_df["asset_type"].isin(["feature_table", "benchmark", "retrieval_index", "embedding_store"]),
    [
        "asset_name",
        "asset_type",
        "priority",
        "freshness_delay_hours",
        "retrieval_freshness_hours",
        "observability_gap",
    ],
].sort_values(["asset_type", "asset_name"]).reset_index(drop=True)
model_facing_df


### What to Notice in the Model-Facing View

This filtered view matters because model-facing assets often fail quietly. If freshness or observability gaps concentrate here, user-visible behavior can change before the platform has enough evidence to explain why.


## Part 5: Compare Benchmark and Retrieval Assets

Benchmarks and retrieval assets fail differently, but prompt-centered systems add a third control surface: runtime context. A useful incident timeline to keep in mind is simple: prompt changes on day 1, behavior shifts on day 2, complaints arrive on day 3, and only then does the team discover it cannot link stored responses back to the prompt version that was live.

In [ ]:
benchmark_and_retrieval_df = assessment_df.loc[
    assessment_df["asset_type"].isin(["benchmark", "retrieval_index", "embedding_store"]),
    [
        "asset_name",
        "asset_type",
        "priority",
        "primary_gap",
        "observability_gap",
        "recommended_action",
    ],
].sort_values(["asset_type", "asset_name"]).reset_index(drop=True)
benchmark_and_retrieval_df


### What to Notice in the Benchmark and Retrieval Comparison

The comparison is useful because it separates two different kinds of trust problem: whether the platform can still compare models fairly, and whether the runtime knowledge state feeding the model is still current and authorized.


## Part 5A: Build a Governance Action Queue

Risk scores become operational only when they turn into an action queue with named authority.

In [ ]:
runtime_asset_types = [
    'prompt_template',
    'context_trace_log',
    'tool_output_log',
    'generated_response_log',
    'retrieval_policy',
]
role_map = {
    'feature_table': 'data owner',
    'benchmark': 'model owner',
    'retrieval_index': 'platform engineer',
    'embedding_store': 'platform engineer',
    'prompt_template': 'product owner',
    'context_trace_log': 'security reviewer',
    'tool_output_log': 'security reviewer',
    'generated_response_log': 'compliance reviewer',
    'retrieval_policy': 'product owner',
}

def governance_action(row):
    if row['priority'] == 'critical':
        return 'block release'
    if row['asset_type'] == 'context_trace_log' and 'retention' in row['recommended_action']:
        return 'redact or archive'
    if row['owner_defined'] == 'no':
        return 'escalate ownership'
    if row['asset_type'] == 'prompt_template' and 'block prompt promotion' in row['recommended_action']:
        return 'block release'
    return 'warn and monitor'

action_queue = assessment_df.loc[
    assessment_df['asset_type'].isin(runtime_asset_types + ['feature_table', 'benchmark', 'retrieval_index', 'embedding_store']),
    ['asset_name', 'asset_type', 'priority', 'primary_gap', 'recommended_action', 'owner_defined']
].copy()
action_queue['role_authority'] = action_queue['asset_type'].map(role_map)
action_queue['governance_action'] = action_queue.apply(governance_action, axis=1)
action_queue.sort_values(['priority', 'asset_name']).reset_index(drop=True)

### Why the Action Queue Matters

Governance is not only scoring risk. It is changing system behavior: warn, block, redact, archive, or escalate before the platform keeps serving model-facing state as if nothing changed.

## Part 5B: False-Green Scenario

The most expensive incidents are often the ones where pipelines look healthy while the model-facing state is already stale.

In [ ]:
false_green_case = pd.DataFrame([
    {
        'system_surface': 'daily pipeline dashboard',
        'status': 'green',
        'hidden_problem': 'support embedding store is 31 hours stale',
        'first_user_visible_symptom': 'assistant cites outdated policy text',
        'control_that_should_act': 'retrieval freshness alert with release-block authority',
    }
])
false_green_case

## Part 5C: Before and After a Control Fix

The notebook should also show that better ownership and retention discipline can change the operational recommendation, not only the narrative.

In [ ]:
before_df = assess_controls(assets_df)
improved_assets = assets_df.copy()
mask = improved_assets['asset_name'] == 'support_context_trace_log'
improved_assets.loc[mask, 'retention_policy_defined'] = 'yes'
improved_assets.loc[mask, 'lineage_coverage_pct'] = 92
improved_assets.loc[mask, 'access_review_age_days'] = 30
after_df = assess_controls(improved_assets)
before_after = before_df.loc[before_df['asset_name'] == 'support_context_trace_log', ['asset_name', 'total_risk_score', 'priority', 'recommended_action']].merge(
    after_df.loc[after_df['asset_name'] == 'support_context_trace_log', ['asset_name', 'total_risk_score', 'priority', 'recommended_action']],
    on='asset_name',
    suffixes=('_before', '_after'),
)
before_after

## Part 6: High-Priority Assets

These are the assets that need the fastest operational response.


In [ ]:
high_priority_df = find_high_priority_assets(assessment_df)
high_priority_df


### How to Read the Control Recommendations

Read the final output as an operational queue, not a moral ranking of datasets. A high-priority asset is one where the control gap is close enough to production behavior that someone needs authority to warn, block, quarantine, redact, roll back, or escalate.


## Part 6A: Inspect Control Artifacts

The chapter names compact control artifacts because governance decisions should be reviewable outside the notebook. Use the assertion and ownership records below as small examples of executable control design.

In [ ]:
quality_artifact_dir = ensure_path_exists(
    COMPANION_ROOT / "artifacts" / "quality",
    "artifact directory",
)
print("data_quality_assertions.yaml")
print("-" * 60)
print("\n".join((ensure_path_exists(quality_artifact_dir / "data_quality_assertions.yaml", "artifact")).read_text(encoding="utf-8").splitlines()[:18]))
print("\nownership_control_record.yaml")
print("-" * 60)
print("\n".join((ensure_path_exists(quality_artifact_dir / "ownership_control_record.yaml", "artifact")).read_text(encoding="utf-8").splitlines()[:18]))
print("\ncontrol_assessment.py")
print("-" * 60)
print("\n".join(module_path.read_text(encoding="utf-8").splitlines()[:18]))

## Exercise

### Concept check
- For each failure type in Chapter 6, separate the quality, security, compliance, and governance question from the action that follows. Name who has authority to warn, block, quarantine, redact, archive, escalate, or roll back.

### Diagnostic scenario
- Use the false-green scenario to explain why observability failed, which retrieval-freshness metric was missing, and whether release should have been blocked.

### Artifact-building exercise
- Build a governance action queue with asset, risk, priority, action, owner, deadline, and escalation path. Include one benchmark asset and one retrieval or prompt-context asset.

### Notebook extension
- Modify one asset so its action changes from block to warn or from warn to proceed, rerun the assessment, and list the evidence bundle a regulator or internal reviewer would still need for one disputed response.
- Then complete `prompt_regression_release_lab.ipynb` for one failed prompt or retrieval case and explain whether the failure should warn, block, pause, or trigger rollback.